# Эксперимент 27: Arabic-speaking children tool

**Статья:** A Novel Speech Analysis and Correction Tool for Arabic-Speaking Children (Новый инструмент анализа и коррекции речи для арабоговорящих детей) 2024

**Ссылка:** [https://arxiv.org/abs/2411.13592](https://arxiv.org/abs/2411.13592)

**Краткое описание модели:** Артикуляционно-ориентированный pipeline: vocal-tract + спектральные полосы/контраст + длительность -> MLP-классификатор.

**Содержание статьи:** Пайплайн ориентирован на инструмент анализа и коррекции речи: выделяются артикуляционные и спектральные характеристики детской речи, после чего обучается нейросетевой классификатор.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, classification_report

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))

from shared import config, data_utils
from shared.results_utils import save_result_csv

## 1. Загрузка данных и разбиение

In [ ]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
print(f"Train: {len(paths_train)}, Val: {len(paths_val)}, Test: {len(paths_test)}")
n_letters = letters_train.shape[1]

Train: 1942, Val: 417, Test: 417


## 2. Подготовка признаков / представлений

In [ ]:
def extractor(path):
    y, sr = data_utils.load_audio(path, sr=config.TARGET_SR, max_sec=config.MAX_DURATION_SEC)
    vt = data_utils.extract_vocal_tract_features(path)
    S = np.abs(librosa.stft(y, n_fft=config.WIN_LENGTH, hop_length=config.HOP_LENGTH)) ** 2
    freqs = librosa.fft_frequencies(sr=sr, n_fft=config.WIN_LENGTH)
    def band(lo, hi):
        m = (freqs >= lo) & (freqs < hi)
        return float(np.mean(S[m])) if np.any(m) else 0.0
    b1, b2, b3, b4 = band(300,900), band(900,1800), band(1800,3200), band(3200,5000)
    cent = librosa.feature.spectral_centroid(S=S, sr=sr, hop_length=config.HOP_LENGTH)[0]
    contrast = librosa.feature.spectral_contrast(S=S, sr=sr)[0]
    extra = np.array([b1,b2,b3,b4,b2/(b3+1e-6),b3/(b4+1e-6),float(np.mean(cent)),float(np.std(cent)),float(np.mean(contrast)),float(np.std(contrast)),len(y)/sr], dtype=np.float32)
    return np.concatenate([vt.astype(np.float32), extra], axis=0)

X_train = data_utils.build_feature_matrix(paths_train, extractor, n_jobs=-1)
X_val   = data_utils.build_feature_matrix(paths_val, extractor, n_jobs=-1)
X_test  = data_utils.build_feature_matrix(paths_test, extractor, n_jobs=-1)

X_train = np.hstack([X_train, letters_train])
X_val   = np.hstack([X_val, letters_val])
X_test  = np.hstack([X_test, letters_test])

## 3. Обучение, оценка и сохранение результата

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()), 
    ("mlp", MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=config.RANDOM_STATE))
])
param_grid = {
    "mlp__alpha": [1e-4, 1e-3, 1e-2], 
    "mlp__learning_rate_init": [1e-3, 3e-4]
}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro", refit=True, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
clf = grid.best_estimator_

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, average="binary", pos_label=config.CLASS_BAD)
roc_auc = roc_auc_score(y_test, y_proba)
precision_bad = precision_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)
recall_bad = recall_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)

print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
display(pd.DataFrame([{"accuracy": accuracy, "f1_macro": f1_macro, "f1_bad": f1_bad, "roc_auc": roc_auc, "precision_bad": precision_bad, "recall_bad": recall_bad}]))

save_result_csv(
    exp_dir=exp_dir, 
    experiment_id="exp_27_arabic_children_tool", 
    experiment_name="Arabic children articulation tool", 
    model="MLP", 
    accuracy=accuracy, 
    f1_macro=f1_macro, 
    f1_bad=f1_bad, 
    roc_auc=roc_auc, 
    precision_bad=precision_bad, 
    recall_bad=recall_bad, 
    notes=f"articulation_tool/mlp_grid={grid.best_params_}", 
    num_params=None, 
    train_time_sec=None
)

Fitting 5 folds for each of 6 candidates, totalling 30 fits


/home/dk/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/dk/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/dk/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


              precision    recall  f1-score   support

        good       0.77      0.77      0.77       282
         bad       0.52      0.52      0.52       135

    accuracy                           0.69       417
   macro avg       0.65      0.65      0.65       417
weighted avg       0.69      0.69      0.69       417



,accuracy,f1_macro,f1_bad,roc_auc,precision_bad,recall_bad
0,0.690647,0.646064,0.520446,0.708878,0.522388,0.518519


Результат сохранён в result.csv текущего эксперимента
